In [117]:
import os
import numpy as np
import pandas as pd

HORIZONS = [5, 10, 21, 63]

# Add models here as they become available
MODEL_NAMES = ["EWMA", "DCC", "HAR"]

PROXY_DIR = "../proxy"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"
N_ASSETS = 8

In [116]:
def load_cov_csv(path):
    return pd.read_csv(path, parse_dates=["Date"]).set_index("Date").sort_index()


def get_proxy_path(h):
    return os.path.join(PROXY_DIR, f"realized_cov_h{h}.csv")


def get_model_path(model_name, h):
    model_name = model_name.upper()

    if model_name == "EWMA":
        return os.path.join(MODELS_DIR, "ewma_cov_lambda094.csv")
    else:
        return os.path.join(RESULTS_DIR, f"{model_name.lower()}_cov_forecast_h{h}.csv")


def load_horizon_data(h, model_names):
    proxy = load_cov_csv(get_proxy_path(h))

    models = {}
    for model_name in model_names:
        path = get_model_path(model_name, h)
        if os.path.exists(path):
            models[model_name] = load_cov_csv(path)
        else:
            print(f"[skip] Missing file for {model_name}, h={h}: {path}")

    return proxy, models

In [118]:
def row_to_matrix(row, n_assets=N_ASSETS):
    mat = row.to_numpy(dtype=float).reshape(n_assets, n_assets)
    mat = 0.5 * (mat + mat.T)
    return mat


def make_spd(mat, eps=1e-10):
    mat = 0.5 * (mat + mat.T)
    mat = mat + eps * np.eye(mat.shape[0])
    return mat


def covariance_to_variance_vector(cov_mat):
    return np.diag(cov_mat)


def covariance_to_correlation_matrix(cov_mat, eps=1e-12):
    cov_mat = 0.5 * (cov_mat + cov_mat.T)
    var = np.diag(cov_mat)
    std = np.sqrt(np.maximum(var, eps))

    corr = cov_mat / np.outer(std, std)
    corr = 0.5 * (corr + corr.T)
    np.fill_diagonal(corr, 1.0)

    return corr

In [119]:
def qlike_loss(proxy_mat, forecast_mat, eps=1e-10):
    S = make_spd(proxy_mat, eps=eps)
    H = make_spd(forecast_mat, eps=eps)

    n = S.shape[0]

    H_inv = np.linalg.inv(H)
    trace_term = np.trace(S @ H_inv)

    sign_s, logdet_s = np.linalg.slogdet(S)
    sign_h, logdet_h = np.linalg.slogdet(H)

    if sign_s <= 0 or sign_h <= 0:
        raise ValueError("Non-positive determinant encountered in QLIKE calculation.")

    return float(trace_term - (logdet_s - logdet_h) - n)


def frobenius_loss(proxy_mat, forecast_mat):
    diff = forecast_mat - proxy_mat
    return float(np.sum(diff ** 2))


def variance_mse_loss(proxy_mat, forecast_mat):
    var_proxy = covariance_to_variance_vector(proxy_mat)
    var_forecast = covariance_to_variance_vector(forecast_mat)
    return float(np.mean((var_forecast - var_proxy) ** 2))


def correlation_mse_loss(proxy_mat, forecast_mat):
    corr_proxy = covariance_to_correlation_matrix(proxy_mat)
    corr_forecast = covariance_to_correlation_matrix(forecast_mat)

    n = corr_proxy.shape[0]
    tri_i, tri_j = np.triu_indices(n, k=1)

    diff = corr_forecast[tri_i, tri_j] - corr_proxy[tri_i, tri_j]
    return float(np.mean(diff ** 2))

In [120]:
def align_forecast_and_proxy(forecast_df, proxy_df, model_name):
    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates].copy()
    proxy_df = proxy_df.loc[common_dates].copy()

    if list(forecast_df.columns) != list(proxy_df.columns):
        raise ValueError(f"Column mismatch between {model_name} and proxy.")

    return forecast_df, proxy_df

In [121]:
def evaluate_against_proxy(forecast_df, proxy_df, model_name, n_assets=N_ASSETS):
    forecast_df, proxy_df = align_forecast_and_proxy(forecast_df, proxy_df, model_name)

    records = []

    for dt in forecast_df.index:
        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets=n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets=n_assets)

        records.append({
            "Date": dt,
            "QLIKE": qlike_loss(S_mat, H_mat),
            "FROBENIUS": frobenius_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_qlike": losses["QLIKE"].mean(),
        "median_qlike": losses["QLIKE"].median(),
        "avg_frobenius": losses["FROBENIUS"].mean()
    })

    return losses, summary


def evaluate_components_against_proxy(forecast_df, proxy_df, model_name, n_assets=N_ASSETS):
    forecast_df, proxy_df = align_forecast_and_proxy(forecast_df, proxy_df, model_name)

    records = []

    for dt in forecast_df.index:
        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets=n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets=n_assets)

        records.append({
            "Date": dt,
            "VAR_MSE": variance_mse_loss(S_mat, H_mat),
            "CORR_MSE": correlation_mse_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_var_mse": losses["VAR_MSE"].mean(),
        "avg_corr_mse": losses["CORR_MSE"].mean()
    })

    return losses, summary

In [122]:
def evaluate_all_models_covariance(models, proxy):
    summaries = []
    losses_dict = {}

    for model_name, forecast_df in models.items():
        losses, summary = evaluate_against_proxy(forecast_df, proxy, model_name=model_name)
        summaries.append(summary)
        losses_dict[model_name] = losses

    summary_table = pd.DataFrame(summaries).reset_index(drop=True)
    return summary_table, losses_dict


def evaluate_all_models_components(models, proxy):
    summaries = []
    losses_dict = {}

    for model_name, forecast_df in models.items():
        losses, summary = evaluate_components_against_proxy(forecast_df, proxy, model_name=model_name)
        summaries.append(summary)
        losses_dict[model_name] = losses

    summary_table = pd.DataFrame(summaries).reset_index(drop=True)
    return summary_table, losses_dict

In [123]:
def build_volatility_regimes(proxy, index, n_assets=N_ASSETS, low_q=0.25, high_q=0.75):
    proxy_eval = proxy.loc[index]

    proxy_vol = proxy_eval.apply(
        lambda row: np.trace(row_to_matrix(row, n_assets=n_assets)),
        axis=1
    )
    proxy_vol.name = "system_volatility"

    low_threshold = proxy_vol.quantile(low_q)
    high_threshold = proxy_vol.quantile(high_q)

    vol_regime = pd.Series(index=proxy_vol.index, dtype="object")
    vol_regime[proxy_vol <= low_threshold] = "low"
    vol_regime[proxy_vol >= high_threshold] = "high"
    vol_regime[(proxy_vol > low_threshold) & (proxy_vol < high_threshold)] = "mid"

    return proxy_vol, vol_regime, low_threshold, high_threshold

In [138]:
def evaluate_regime_covariance(cov_losses_dict, vol_regime):
    tables = []

    for model_name, losses_df in cov_losses_dict.items():
        temp = losses_df.copy()
        temp["regime"] = vol_regime.loc[temp.index]

        grouped = temp.groupby("regime")[["QLIKE", "FROBENIUS"]].mean()
        grouped["model"] = model_name
        grouped = grouped.reset_index()

        tables.append(grouped)

    regime_table = pd.concat(tables, axis=0).reset_index(drop=True)
    regime_table = regime_table[["model", "regime", "QLIKE", "FROBENIUS"]]
    return regime_table


def evaluate_regime_components(component_losses_dict, vol_regime):
    tables = []

    for model_name, losses_df in component_losses_dict.items():
        temp = losses_df.copy()
        temp["regime"] = vol_regime.loc[temp.index]

        grouped = temp.groupby("regime")[["VAR_MSE", "CORR_MSE"]].mean()
        grouped["model"] = model_name
        grouped = grouped.reset_index()

        tables.append(grouped)

    regime_table = pd.concat(tables, axis=0).reset_index(drop=True)
    regime_table = regime_table[["model", "regime", "VAR_MSE", "CORR_MSE"]]
    return regime_table

In [137]:
def run_full_evaluation_for_horizon(h, model_names, regime_order=("low", "mid", "high")):
    """
    Run full evaluation pipeline for one horizon h.

    Returns a dictionary with:
    - proxy
    - models
    - summary tables
    - loss dictionaries
    - volatility regime objects
    - wide-format display tables
    """

    # -----------------------------
    # 1) Load data
    # -----------------------------
    proxy, models = load_horizon_data(h=h, model_names=model_names)

    if len(models) == 0:
        raise ValueError(f"No model files found for horizon h={h}.")

    # -----------------------------
    # 2) Full-sample evaluation
    # -----------------------------
    cov_summary_table, cov_losses_dict = evaluate_all_models_covariance(models, proxy)
    component_summary_table, component_losses_dict = evaluate_all_models_components(models, proxy)

    # -----------------------------
    # 3) Build volatility regimes
    # -----------------------------
    common_index = next(iter(cov_losses_dict.values())).index
    proxy_vol, vol_regime, low_threshold, high_threshold = build_volatility_regimes(proxy, common_index)

    # -----------------------------
    # 4) Regime evaluation
    # -----------------------------
    cov_regime_table = evaluate_regime_covariance(cov_losses_dict, vol_regime)
    component_regime_table = evaluate_regime_components(component_losses_dict, vol_regime)

    # -----------------------------
    # 5) Wide tables for display
    # -----------------------------
    cov_overall_wide = (
        cov_summary_table
        .set_index("model")[["avg_qlike", "avg_frobenius"]]
        .rename(columns={
            "avg_qlike": "QLIKE",
            "avg_frobenius": "FROBENIUS"
        })
        .T
    )

    component_overall_wide = (
        component_summary_table
        .set_index("model")[["avg_var_mse", "avg_corr_mse"]]
        .rename(columns={
            "avg_var_mse": "VAR_MSE",
            "avg_corr_mse": "CORR_MSE"
        })
        .T
    )

    qlike_regime_wide = (
        cov_regime_table
        .pivot(index="regime", columns="model", values="QLIKE")
        .reindex(regime_order)
    )

    frobenius_regime_wide = (
        cov_regime_table
        .pivot(index="regime", columns="model", values="FROBENIUS")
        .reindex(regime_order)
    )

    var_regime_wide = (
        component_regime_table
        .pivot(index="regime", columns="model", values="VAR_MSE")
        .reindex(regime_order)
    )

    corr_regime_wide = (
        component_regime_table
        .pivot(index="regime", columns="model", values="CORR_MSE")
        .reindex(regime_order)
    )

    # -----------------------------
    # 6) Collect everything
    # -----------------------------
    results = {
        "h": h,
        "proxy": proxy,
        "models": models,

        "cov_summary_table": cov_summary_table,
        "component_summary_table": component_summary_table,
        "cov_regime_table": cov_regime_table,
        "component_regime_table": component_regime_table,

        "cov_losses_dict": cov_losses_dict,
        "component_losses_dict": component_losses_dict,

        "proxy_vol": proxy_vol,
        "vol_regime": vol_regime,
        "low_threshold": low_threshold,
        "high_threshold": high_threshold,

        "cov_overall_wide": cov_overall_wide,
        "component_overall_wide": component_overall_wide,
        "qlike_regime_wide": qlike_regime_wide,
        "frobenius_regime_wide": frobenius_regime_wide,
        "var_regime_wide": var_regime_wide,
        "corr_regime_wide": corr_regime_wide
    }

    return results

In [140]:
def display_evaluation_results(results):

    from IPython.display import display

    h = results["h"]

    print(f"\n================ HORIZON h={h} ================\n")

    print("========== COVARIANCE PERFORMANCE ==========")
    display(results["cov_overall_wide"])

    print("\n========== VARIANCE / CORRELATION PERFORMANCE ==========")
    display(results["component_overall_wide"])

    print("\n========== QLIKE BY VOLATILITY REGIME ==========")
    display(results["qlike_regime_wide"])

    print("\n========== FROBENIUS BY VOLATILITY REGIME ==========")
    display(results["frobenius_regime_wide"])

    print("\n========== VARIANCE MSE BY REGIME ==========")
    display(results["var_regime_wide"])

    print("\n========== CORRELATION MSE BY REGIME ==========")
    display(results["corr_regime_wide"])

    print("\n========== VOLATILITY REGIME THRESHOLDS ==========")
    print("Low threshold :", results["low_threshold"])
    print("High threshold:", results["high_threshold"])
    print(results["vol_regime"].value_counts())

In [129]:
cov_summary_table

,model,n_obs,start_date,end_date,avg_qlike,median_qlike,avg_frobenius
0,EWMA,1235,2021-01-04,2025-12-02,5.235591,4.336692,0.000002
1,DCC,1235,2021-01-04,2025-12-02,4.488435,3.783640,0.000017
2,HAR,1235,2021-01-04,2025-12-02,6.543897,4.800582,0.000002


In [130]:
component_summary_table

,model,n_obs,start_date,end_date,avg_var_mse,avg_corr_mse
0,EWMA,1235,2021-01-04,2025-12-02,1.491691e-07,0.080336
1,DCC,1235,2021-01-04,2025-12-02,2.108982e-06,0.075739
2,HAR,1235,2021-01-04,2025-12-02,1.520447e-07,0.097812


In [132]:
cov_regime_table


,model,regime,QLIKE,FROBENIUS
0,EWMA,high,7.515286,3.213900e-06
1,EWMA,low,4.218387,6.027697e-07
2,EWMA,mid,4.603322,1.211043e-06
3,DCC,high,6.111136,5.643577e-06
4,DCC,low,4.214621,4.841687e-05
5,DCC,mid,3.812899,7.680796e-06
6,HAR,high,10.245549,4.415299e-06
7,HAR,low,4.654857,4.425169e-07
8,HAR,mid,5.636122,7.724633e-07


In [133]:
component_regime_table

,model,regime,VAR_MSE,CORR_MSE
0,EWMA,high,3.113609e-07,0.101057
1,EWMA,low,5.092294e-08,0.073199
2,EWMA,mid,1.171445e-07,0.073534
3,DCC,high,6.091462e-07,0.096769
4,DCC,low,5.976040e-06,0.075213
5,DCC,mid,9.234530e-07,0.065470
6,HAR,high,4.456766e-07,0.138734
7,HAR,low,3.497984e-08,0.065545
8,HAR,mid,6.361806e-08,0.093478


In [142]:
H = 21

results_21 = run_full_evaluation_for_horizon(
    h=H,
    model_names=MODEL_NAMES
)

display_evaluation_results(results_21)


================ HORIZON h=21 ================

========== COVARIANCE PERFORMANCE ==========


model,EWMA,DCC,HAR
QLIKE,5.235591,4.488435,6.543897
FROBENIUS,0.000002,0.000017,0.000002



========== VARIANCE / CORRELATION PERFORMANCE ==========


model,EWMA,DCC,HAR
VAR_MSE,1.491691e-07,0.000002,1.520447e-07
CORR_MSE,8.033628e-02,0.075739,9.781215e-02



========== QLIKE BY VOLATILITY REGIME ==========


...


========== VARIANCE MSE BY REGIME ==========


model,DCC,EWMA,HAR
regime,,,
low,5.976040e-06,5.092294e-08,3.497984e-08
mid,9.234530e-07,1.171445e-07,6.361806e-08
high,6.091462e-07,3.113609e-07,4.456766e-07



========== CORRELATION MSE BY REGIME ==========


model,DCC,EWMA,HAR
regime,,,
low,0.075213,0.073199,0.065545
mid,0.065470,0.073534,0.093478
high,0.096769,0.101057,0.138734



========== VOLATILITY REGIME THRESHOLDS ==========
Low threshold : 0.001117003960638956
High threshold: 0.0024032604891289543
mid     617
high    309
low     309
Name: count, dtype: int64


In [102]:
print("Proxy shape:", proxy.shape)
print("EWMA shape:", ewma.shape)
print("DCC shape:", dcc.shape)
print("HAR shape:", har.shape)

print("Proxy vs EWMA same columns:", list(proxy.columns) == list(ewma.columns))
print("Proxy vs DCC same columns:", list(proxy.columns) == list(dcc.columns))
print("Proxy vs HAR same columns:", list(proxy.columns) == list(har.columns))


common_ewma = proxy.index.intersection(ewma.index)
common_dcc  = proxy.index.intersection(dcc.index)
common_har  = proxy.index.intersection(har.index)

print("EWMA common dates:", len(common_ewma), common_ewma.min().date(), "->", common_ewma.max().date())
print("DCC common dates:", len(common_dcc), common_dcc.min().date(), "->", common_dcc.max().date())
print("HAR common dates:", len(common_har), common_har.min().date(), "->", common_har.max().date())

Proxy shape: (2184, 64)
EWMA shape: (1255, 64)
DCC shape: (1255, 64)
HAR shape: (1235, 64)
Proxy vs EWMA same columns: True
Proxy vs DCC same columns: True
Proxy vs HAR same columns: True
EWMA common dates: 1235 2021-01-04 -> 2025-12-02
DCC common dates: 1235 2021-01-04 -> 2025-12-02
HAR common dates: 1235 2021-01-04 -> 2025-12-02


In [103]:
def row_to_matrix(row, n_assets=8):
    mat = row.to_numpy(dtype=float).reshape(n_assets, n_assets)
    mat = 0.5 * (mat + mat.T)
    return mat

def make_spd(mat, eps=1e-10):
    mat = 0.5 * (mat + mat.T)
    mat = mat + eps * np.eye(mat.shape[0])
    return mat

def qlike_loss(proxy_mat, forecast_mat, eps=1e-10):
    S = make_spd(proxy_mat, eps=eps)
    H = make_spd(forecast_mat, eps=eps)

    n = S.shape[0]

    H_inv = np.linalg.inv(H)
    trace_term = np.trace(S @ H_inv)

    sign_s, logdet_s = np.linalg.slogdet(S)
    sign_h, logdet_h = np.linalg.slogdet(H)

    if sign_s <= 0 or sign_h <= 0:
        raise ValueError("Non-positive determinant encountered in QLIKE calculation.")

    return float(trace_term - (logdet_s - logdet_h) - n)

def frobenius_loss(proxy_mat, forecast_mat):
    diff = forecast_mat - proxy_mat
    return float(np.sum(diff ** 2))

def evaluate_against_proxy(forecast_df, proxy_df, model_name, n_assets=8):
    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates].copy()
    proxy_df = proxy_df.loc[common_dates].copy()

    if list(forecast_df.columns) != list(proxy_df.columns):
        raise ValueError(f"Column mismatch between {model_name} and proxy.")

    records = []

    for dt in common_dates:
        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets=n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets=n_assets)

        records.append({
            "Date": dt,
            "QLIKE": qlike_loss(S_mat, H_mat),
            "FROBENIUS": frobenius_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_qlike": losses["QLIKE"].mean(),
        "avg_frobenius": losses["FROBENIUS"].mean()
    })

    return losses, summary

In [104]:
ewma_losses, ewma_summary = evaluate_against_proxy(ewma, proxy, model_name="EWMA")
dcc_losses, dcc_summary = evaluate_against_proxy(dcc, proxy, model_name="DCC")
har_losses, har_summary = evaluate_against_proxy(har, proxy, model_name="HAR")

summary_table = pd.DataFrame([ewma_summary, dcc_summary,har_summary]).reset_index(drop=True)
summary_table

,model,n_obs,start_date,end_date,avg_qlike,avg_frobenius
0,EWMA,1235,2021-01-04,2025-12-02,5.235591,0.000002
1,DCC,1235,2021-01-04,2025-12-02,4.488435,0.000017
2,HAR,1235,2021-01-04,2025-12-02,6.495903,0.000002


In [105]:
loss_compare = pd.DataFrame({
    "EWMA_QLIKE": ewma_losses["QLIKE"],
    "DCC_QLIKE": dcc_losses["QLIKE"],
    "HAR_QLIKE": har_losses["QLIKE"],
    "EWMA_FROB": ewma_losses["FROBENIUS"],
    "DCC_FROB": dcc_losses["FROBENIUS"],
    "HAR_FROB": har_losses["FROBENIUS"],
})

loss_compare.describe()

qlike_compare = pd.DataFrame({
    "EWMA": ewma_losses["QLIKE"],
    "DCC": dcc_losses["QLIKE"],
    "HAR": har_losses["QLIKE"]
})

print("Best model by QLIKE share:")
print(qlike_compare.idxmin(axis=1).value_counts(normalize=True))

Best model by QLIKE share:
DCC     0.635628
EWMA    0.222672
HAR     0.141700
Name: proportion, dtype: float64


Varians and correlation performance


In [106]:
def covariance_to_variance_vector(cov_mat):
    return np.diag(cov_mat)


def covariance_to_correlation_matrix(cov_mat, eps=1e-12):
    cov_mat = 0.5 * (cov_mat + cov_mat.T)

    var = np.diag(cov_mat)
    std = np.sqrt(np.maximum(var, eps))

    corr = cov_mat / np.outer(std, std)

    corr = 0.5 * (corr + corr.T)
    np.fill_diagonal(corr, 1.0)

    return corr

In [107]:
def variance_mse_loss(proxy_mat, forecast_mat):

    var_proxy = covariance_to_variance_vector(proxy_mat)
    var_forecast = covariance_to_variance_vector(forecast_mat)

    return float(np.mean((var_forecast - var_proxy) ** 2))


def correlation_mse_loss(proxy_mat, forecast_mat):

    corr_proxy = covariance_to_correlation_matrix(proxy_mat)
    corr_forecast = covariance_to_correlation_matrix(forecast_mat)

    n = corr_proxy.shape[0]

    tri_i, tri_j = np.triu_indices(n, k=1)

    diff = corr_forecast[tri_i, tri_j] - corr_proxy[tri_i, tri_j]

    return float(np.mean(diff ** 2))

In [108]:
def evaluate_components_against_proxy(forecast_df, proxy_df, model_name, n_assets=8):

    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates].copy()
    proxy_df = proxy_df.loc[common_dates].copy()

    if list(forecast_df.columns) != list(proxy_df.columns):
        raise ValueError(f"Column mismatch between {model_name} and proxy.")

    records = []

    for dt in common_dates:

        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets=n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets=n_assets)

        records.append({
            "Date": dt,
            "VAR_MSE": variance_mse_loss(S_mat, H_mat),
            "CORR_MSE": correlation_mse_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_var_mse": losses["VAR_MSE"].mean(),
        "avg_corr_mse": losses["CORR_MSE"].mean()
    })

    return losses, summary

In [109]:
ewma_comp_losses, ewma_comp_summary = evaluate_components_against_proxy(
    ewma, proxy, model_name="EWMA"
)

dcc_comp_losses, dcc_comp_summary = evaluate_components_against_proxy(
    dcc, proxy, model_name="DCC"
)

har_comp_losses, har_comp_summary = evaluate_components_against_proxy(
    har, proxy, model_name="HAR"
)

component_summary_table = pd.DataFrame(
    [ewma_comp_summary, dcc_comp_summary, har_comp_summary]
).reset_index(drop=True)

component_summary_table

,model,n_obs,start_date,end_date,avg_var_mse,avg_corr_mse
0,EWMA,1235,2021-01-04,2025-12-02,1.491691e-07,0.080336
1,DCC,1235,2021-01-04,2025-12-02,2.108982e-06,0.075739
2,HAR,1235,2021-01-04,2025-12-02,1.518580e-07,0.097724


Høy/lav volatilitet 

In [110]:
proxy_test = proxy.loc[ewma_losses.index]

proxy_vol = proxy_test.apply(
    lambda row: np.trace(row_to_matrix(row)), axis=1
)

proxy_vol.name = "system_volatility"

proxy_vol.describe()

count    1235.000000
mean        0.001927
std         0.001188
min         0.000348
25%         0.001117
50%         0.001560
75%         0.002403
max         0.006489
Name: system_volatility, dtype: float64

In [111]:
low_threshold = proxy_vol.quantile(0.25)
high_threshold = proxy_vol.quantile(0.75)

print("Low volatility threshold:", low_threshold)
print("High volatility threshold:", high_threshold)

vol_regime = pd.Series(index=proxy_vol.index, dtype="object")

vol_regime[proxy_vol <= low_threshold] = "low"
vol_regime[proxy_vol >= high_threshold] = "high"
vol_regime[(proxy_vol > low_threshold) & (proxy_vol < high_threshold)] = "mid"

vol_regime.value_counts()

Low volatility threshold: 0.001117003960638956
High volatility threshold: 0.0024032604891289543


mid     617
high    309
low     309
Name: count, dtype: int64

In [112]:
ewma_losses["regime"] = vol_regime.loc[ewma_losses.index]
dcc_losses["regime"] = vol_regime.loc[dcc_losses.index]
har_losses["regime"] = vol_regime.loc[har_losses.index]

ewma_comp_losses["regime"] = vol_regime.loc[ewma_comp_losses.index]
dcc_comp_losses["regime"] = vol_regime.loc[dcc_comp_losses.index]
har_comp_losses["regime"] = vol_regime.loc[har_comp_losses.index]


qlike_regime_table = pd.DataFrame({
    "EWMA": ewma_losses.groupby("regime")["QLIKE"].mean(),
    "DCC": dcc_losses.groupby("regime")["QLIKE"].mean(),
    "HAR": har_losses.groupby("regime")["QLIKE"].mean()
})

qlike_regime_table

frob_regime_table = pd.DataFrame({
    "EWMA": ewma_losses.groupby("regime")["FROBENIUS"].mean(),
    "DCC": dcc_losses.groupby("regime")["FROBENIUS"].mean(),
    "HAR": har_losses.groupby("regime")["FROBENIUS"].mean()
})

frob_regime_table

var_regime_table = pd.DataFrame({
    "EWMA": ewma_comp_losses.groupby("regime")["VAR_MSE"].mean(),
    "DCC": dcc_comp_losses.groupby("regime")["VAR_MSE"].mean(),
    "HAR": har_comp_losses.groupby("regime")["VAR_MSE"].mean()
})

var_regime_table

corr_regime_table = pd.DataFrame({
    "EWMA": ewma_comp_losses.groupby("regime")["CORR_MSE"].mean(),
    "DCC": dcc_comp_losses.groupby("regime")["CORR_MSE"].mean(),
    "HAR": har_comp_losses.groupby("regime")["CORR_MSE"].mean()
})

corr_regime_table

,EWMA,DCC,HAR
regime,,,
high,0.101057,0.096769,0.137959
low,0.073199,0.075213,0.065651
mid,0.073534,0.065470,0.093636


In [113]:
regime_summary = pd.concat(
    {
        "QLIKE": qlike_regime_table,
        "FROBENIUS": frob_regime_table,
        "VAR_MSE": var_regime_table,
        "CORR_MSE": corr_regime_table
    },
    axis=1
)

regime_summary

QLIKE                          FROBENIUS                          \
            EWMA       DCC        HAR          EWMA       DCC           HAR   
regime                                                                        
high    7.515286  6.111136  10.128875  3.213900e-06  0.000006  4.435080e-06   
low     4.218387  4.214621   4.658417  6.027697e-07  0.000048  4.480275e-07   
mid     4.603322  3.812899   5.596705  1.211043e-06  0.000008  7.460858e-07   

             VAR_MSE                              CORR_MSE                      
                EWMA           DCC           HAR      EWMA       DCC       HAR  
regime                                                                          
high    3.113609e-07  6.091462e-07  4.484883e-07  0.101057  0.096769  0.137959  
low     5.092294e-08  5.976040e-06  3.535433e-08  0.073199  0.075213  0.065651  
mid     1.171445e-07  9.234530e-07  6.164882e-08  0.073534  0.065470  0.093636

In [114]:
def system_volatility(df, n_assets=8):
    vol = df.apply(
        lambda row: np.sqrt(np.trace(row_to_matrix(row, n_assets))),
        axis=1
    )
    return vol

realized_vol = system_volatility(proxy)
ewma_vol = system_volatility(ewma)
dcc_vol = system_volatility(dcc)
har_vol = system_volatility(har)

vol_df = pd.DataFrame({
    "Realized": realized_vol,
    "EWMA": ewma_vol,
    "DCC": dcc_vol,
    "HAR": har_vol
}).dropna()

vol_df.head()

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["Realized"],
    mode="lines",
    name="Realized volatility",
    line=dict(width=2)
))

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["EWMA"],
    mode="lines",
    name="EWMA forecast",
    line=dict(width=1)
))

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["DCC"],
    mode="lines",
    name="DCC forecast",
    line=dict(width=1)
))

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["HAR"],
    mode="lines",
    name="HAR forecast",
    line=dict(width=1)
))

fig.update_layout(
    title="System Volatility: Realized vs Forecast",
    xaxis_title="Date",
    yaxis_title="Volatility",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()